# Importing Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split

In [ ]:
import tensorflow as tf
from tensorflow.keras import backend as K
from tensorflow.keras.models import Model, Sequential
from tensorflow.keras.layers import *
from tensorflow.keras.layers import TimeDistributed
from tensorflow.keras.layers import Embedding
from tensorflow.keras.layers import Lambda, Permute
from tensorflow.keras.optimizers import Adam, RMSprop
from tensorflow.keras import regularizers
from tensorflow.keras.models import load_model
from tensorflow.keras.callbacks import TensorBoard
# from tensorflow.keras.layers import *
from keras.utils.np_utils import to_categorical

In [ ]:
df1 = pd.read_csv('/content/SpectralFeatures.csv')
df2 = pd.read_csv('/content/CepstralFeatures.csv')
df3 = pd.read_csv('/content/GFCC_Features.csv')


# Feature Selection

## Spectral Features

In [ ]:
df1

In [ ]:
df1.drop('Unnamed: 0',axis=1,inplace=True)

In [ ]:
le1 = LabelEncoder()
df1['Species'] = le1.fit_transform(df3['Species'])

In [ ]:
corr = df1.corr()
l = corr['Species']

In [ ]:
l

In [ ]:
index = []
for i in range(len(l)):
  if l[i]>0.1 or l[i]<-0.1:
    index.append(i)

In [ ]:
index

In [ ]:
col = df1.columns[index]

In [ ]:
col

In [ ]:
train1 = df1[col]

## Cepstral Features

In [ ]:
df2

In [ ]:
df2.drop('Unnamed: 0',axis=1,inplace=True)

In [ ]:
df2

In [ ]:
le2 = LabelEncoder()
df2['Species'] = le2.fit_transform(df3['Species'])

In [ ]:
corr = df2.corr()

In [ ]:
l = corr['Species']
l

In [ ]:
index = []
for i in range(len(l)):
  if l[i]>0.1 or l[i]<-0.1:
    index.append(i)

In [ ]:
index

In [ ]:
feat = df2.columns[index]

In [ ]:
feat

In [ ]:
train2 = df2[feat]

In [ ]:
train2

## GFCC Features

In [ ]:
df3

In [ ]:
df3.drop('Unnamed: 0',axis=1,inplace=True)

In [ ]:
le3 = LabelEncoder()
df3['Species'] = le3.fit_transform(df3['Species'])

In [ ]:
corr = df3.corr()
l = corr['Species']

In [ ]:
index = []
for i in range(len(l)):
  if l[i]>0.1 or l[i]<-0.1:
    index.append(i)

In [ ]:
index

In [ ]:
d = df3.columns[index]

In [ ]:
d

In [ ]:
train3 = df3[d]

In [ ]:
train3

## Combining Best Features i.e. most correlated features of each dataframe

In [ ]:
df1.drop('Unnamed: 0',axis=1,inplace=True)
df2.drop('Unnamed: 0',axis=1,inplace=True)
df3.drop('Unnamed: 0',axis=1,inplace=True)

In [ ]:
df4 = pd.merge(df1, df2, on=["Id", "Species"])

In [ ]:
df4.shape

In [ ]:
df4 = pd.merge(df4,df3,on=["Id", "Species"])

In [ ]:
df4

In [ ]:
l = ['zero_crossing_rate', 'spectral_centroids', 'spectral_rolloff',
       'spectral_flux', 'chroma11', 'mfcc0', 'mfcc_delta_0', 'mfcc1', 'mfcc2', 'mfcc3',
       'mfcc_delta_3', 'mfcc4', 'mfcc5', 'mfcc7', 'mfcc10', 'mfcc_accelerate_10', 'mfcc11', 'mfcc12','gfcc4', 'gfcc7', 'gfcc13', 'Species']

In [ ]:
len(l)

In [ ]:
train = df4[l]

In [ ]:
train

In [ ]:
train.to_csv('BestFeaturesSetOnCorrelation.csv')

# Model Estimation

In [ ]:
train = pd.read_csv('/content/BestFeaturesSetOnCorrelation.csv')

In [ ]:
train.drop('Unnamed: 0',axis=1,inplace=True)

In [ ]:
X  = train.drop('Species',axis=1)
y = train['Species']

In [ ]:
le = LabelEncoder()
y = le.fit_transform(y)

In [ ]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.35,shuffle=True)

In [ ]:
sc = StandardScaler()
X_train = sc.fit_transform(X_train)
X_test = sc.fit_transform(X_test)

In [ ]:
X_test.shape

In [ ]:
X_train = np.array(X_train)
y_train = np.array(y_train)
X_test = np.array(X_test)
y_test = np.array(y_test)

In [ ]:
y_train

In [ ]:
from tensorflow.keras.layers import *
from keras.utils.np_utils import to_categorical
from sklearn import metrics

In [ ]:
y_train = to_categorical(y_train)
y_test = to_categorical(y_test)

In [ ]:
y_train.shape

In [ ]:
def get_model(inp_dim=21):
    # build model
    model = Sequential()
    model.add(Dense(256, input_shape=(inp_dim, )))
    model.add(Activation('relu'))
    # model.add(Dropout(0.5))
    # model.add(Dense(256))
    # model.add(Activation('relu'))
    # model.add(Dropout(0.5))
    model.add(Dense(128))
    model.add(Activation('relu'))
    # model.add(Dropout(0.5))
    model.add(Dense(128))
    model.add(Activation('relu'))
    # model.add(Dropout(0.5))
    # model.add(Dense(64))
    # model.add(Activation('relu'))
    # model.add(Dropout(0.5))
    model.add(Dense(32))
    model.add(Activation('relu'))
    model.add(Dropout(0.5))
    model.add(Dense(9, kernel_regularizer=regularizers.l2(0.01)))
    model.add(Activation('softmax'))

    model.compile(loss='categorical_crossentropy', optimizer=Adam(learning_rate=1e-4),metrics=['accuracy'])
    # model.summary()
    return model

In [ ]:
from tensorflow.keras.callbacks import Callback

class StopTrainingCallback(Callback):
    def on_epoch_end(self, epoch, logs=None):
        if logs is None:
            logs = {}
        if logs.get('val_accuracy') >= 0.97 and logs.get('val_loss')<=0.25:
            print('\nValidation accuracy has reached 97%, stopping training...')
            self.model.stop_training = True

In [ ]:
model1 = get_model()

In [ ]:
model_history=model1.fit(X_train, y_train, batch_size=16, epochs=75, validation_data=(X_test, y_test), verbose=2, callbacks=[StopTrainingCallback()])

In [ ]:
test_loss, test_acc = model1.evaluate(X_test, y_test)
print('Test accuracy:', test_acc)

In [ ]:
loss = model_history.history['loss']
val_loss = model_history.history['val_loss']
epochs = range(1, len(loss)+1)

In [ ]:
plt.plot(epochs,loss,label='training_loss')
plt.plot(epochs,val_loss,label='val_loss')
plt.show()

In [ ]:
from tensorflow.keras.models import save_model
save_model(model1, "model.h5")